# FlavourNet: from coffee shop reviews to a knowledge graph

This notebook turns 10,000 Yelp reviews of Coffee & Tea businesses into a network of concepts (drinks, food, service, ambience, quality words) and the relationships customers draw between them, then analyses that network.

Pipeline: **preprocess → extract concepts (TF-IDF, YAKE, spaCy NER) → extract relationships (dependency parsing) → filter by PMI → build the graph → centrality, communities and sentiment**.

Input: `data/coffee_tea_sample.csv`, built from the Yelp Open Dataset by `python src/build_sample.py` (the reviews themselves are not redistributed; see the README).

In [ ]:
import math
import re
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import community as community_louvain  # python-louvain
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import spacy
import yake
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
from wordcloud import WordCloud

warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, OUTPUTS = ROOT / "data", ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

nlp = spacy.load("en_core_web_sm")
SEED = 42

## 1. Load and preprocess the reviews

Lower-case, strip everything that is not a letter, then lemmatise with spaCy and drop stop words and tokens of two letters or fewer. `clean_text` feeds TF-IDF; the raw text is kept for YAKE, NER and dependency parsing, which need real sentences.

In [ ]:
reviews = pd.read_csv(DATA / "coffee_tea_sample.csv")
print(f"{len(reviews):,} reviews from {reviews.business_name.nunique():,} businesses in "
      f"{reviews.state.nunique()} states, {reviews.date.min()[:4]}-{reviews.date.max()[:4]}")
reviews.stars.value_counts().sort_index()

In [ ]:
def preprocess(doc):
    return [t.lemma_ for t in doc
            if not t.is_stop and not t.is_punct and not t.is_space and t.is_alpha and len(t.text) > 2]

cleaned = (re.sub(r"[^a-zA-Z\s]", " ", str(t).lower()) for t in reviews.text)
reviews["tokens"] = [preprocess(doc) for doc in nlp.pipe(cleaned, batch_size=200)]
reviews["clean_text"] = reviews.tokens.str.join(" ")
print("Example tokens:", reviews.tokens[0][:10])

## 2. Concept extraction

Three complementary methods, combined into one vocabulary of concepts:

- **TF-IDF** (top 300 unigrams and bigrams) finds terms that characterise individual reviews
- **YAKE** finds key phrases in each raw review without needing a corpus; the 100 most frequent are kept
- **spaCy NER** picks up named things: brands, places and products (top 50)

In [ ]:
tfidf = TfidfVectorizer(max_features=300, ngram_range=(1, 2), min_df=5, max_df=0.85)
tfidf_matrix = tfidf.fit_transform(reviews.clean_text).tocsr()
terms = tfidf.get_feature_names_out()
tfidf_keywords = set(terms)

tfidf_per_review = pd.DataFrame(
    [{"review_id": reviews.review_id.iat[i], "concept": terms[j], "tfidf_score": round(s, 6)}
     for i in range(tfidf_matrix.shape[0])
     for j, s in zip(tfidf_matrix[i].indices, tfidf_matrix[i].data)])
tfidf_global = (tfidf_per_review.groupby("concept").tfidf_score.mean()
                .sort_values(ascending=False).rename("mean_tfidf").reset_index())
concept_importance = dict(zip(tfidf_global.concept, tfidf_global.mean_tfidf))

print(f"TF-IDF concepts: {len(tfidf_keywords)}")
print("Highest mean TF-IDF:", tfidf_global.concept.head(15).tolist())

In [ ]:
kw_extractor = yake.KeywordExtractor(lan="en", n=2, dedupLim=0.7, top=15)
yake_df = pd.DataFrame(
    [{"review_id": rid, "concept": kw.lower(), "yake_importance": round(1 - score, 4)}
     for rid, text in zip(reviews.review_id, reviews.text)
     for kw, score in kw_extractor.extract_keywords(str(text))])
yake_freq = Counter(yake_df.concept)
top_yake = {kw for kw, _ in yake_freq.most_common(100)}
print(f"YAKE concepts: {len(top_yake)}")
pd.Series(dict(yake_freq.most_common(15)), name="reviews").to_frame().T

In [ ]:
NER_LABELS = {"ORG", "PRODUCT", "GPE", "FAC", "NORP"}
reviews["ner_entities"] = [[(e.text.lower(), e.label_) for e in doc.ents if e.label_ in NER_LABELS]
                           for doc in nlp.pipe(reviews.text.astype(str), batch_size=200)]
ner_freq = Counter(e for ents in reviews.ner_entities for e, _ in ents)
top_ner = {e for e, _ in ner_freq.most_common(50)}

all_concepts = tfidf_keywords | top_yake | top_ner
token_freq = Counter(t for tokens in reviews.tokens for t in tokens)
print(f"NER entities: {len(top_ner)}  ->  combined vocabulary: {len(all_concepts)} concepts")

Each concept gets a type from a hand-built coffee-shop lexicon. Anything not in the lexicon is a general `CONCEPT`.

In [ ]:
ENTITY_CATEGORIES = {
    "DRINK": {"coffee", "espresso", "latte", "cappuccino", "matcha", "tea", "americano", "mocha", "chai",
              "cortado", "flat white", "macchiato", "frappuccino", "cold brew", "iced coffee", "boba", "taro",
              "hot chocolate", "juice", "smoothie"},
    "FOOD": {"pastry", "croissant", "bagel", "sandwich", "cake", "muffin", "toast", "cookie", "waffle", "scone",
             "pancake", "egg", "donut", "biscuit", "quiche", "granola", "oatmeal", "brownie", "salad", "wrap",
             "avocado", "cream", "cheese", "bread", "chicken"},
    "SERVICE": {"barista", "staff", "service", "server", "cashier", "friendly", "attentive", "helpful",
                "efficient", "rude", "slow", "fast", "wait", "queue", "order"},
    "AMBIENCE": {"atmosphere", "ambience", "cozy", "noisy", "quiet", "seating", "decor", "music", "lighting",
                 "outdoor", "indoor", "wifi", "space", "parking", "clean", "busy", "crowded"},
    "EXPERIENCE": {"work", "study", "morning", "weekend", "visit", "brunch", "lunch", "breakfast", "relax",
                   "hangout", "date", "meeting"},
    "QUALITY": {"good", "great", "delicious", "amazing", "fresh", "hot", "cold", "sweet", "excellent",
                "awesome", "perfect", "strong", "weak", "expensive", "cheap", "bad", "terrible", "poor", "warm",
                "bitter", "tasty", "wonderful", "crispy", "fluffy", "creamy", "rich", "smooth", "bold", "mild",
                "overpriced", "affordable", "outstanding", "mediocre", "decent", "disappointing"},
}
CONCEPT_TYPE = {w: cat for cat, words in ENTITY_CATEGORIES.items() for w in words}

def get_type(concept):
    return CONCEPT_TYPE.get(concept.lower(), "CONCEPT")

concepts_df = (pd.DataFrame([{"concept": c, "frequency": token_freq.get(c, 0), "entity_type": get_type(c)}
                             for c in all_concepts])
               .sort_values("frequency", ascending=False, ignore_index=True))
concepts_df.entity_type.value_counts()